# Cifar LTM Evaluate
This notebook is used to evaluate a pretrained (or simply trained) LTM 
on different sets of coarse and fine classes.

In [ ]:
%load_ext autoreload
%autoreload 2

from dataclasses import dataclass

import torch
import torch.nn.functional as F
from model.resnet import ResNetConfig
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

from environment.cifar.cifar_classifier import CifarClassifier
from environment.cifar.cifar_dataset import Cifar100Dataset
from util.log import get_run_path

run_root_path = "cifar_100_evaluate"
run_path = get_run_path(
    prefix = run_root_path, 
    path = "./runs",
)
data_file_path = "/home/dave/dev/cifar-100-python"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

writer = SummaryWriter(log_dir=run_path)

In [ ]:
# Test that all 100 classes have indeed been mapped
all_classes = Cifar100Dataset.get_fine_classes([1,2,3,4,5])
print(len(all_classes))
for i in range(100):
    if i not in all_classes:
        print(f"Error: {i} not in all classes.")


In [ ]:
@dataclass
class EpochMetrics:
    mean_loss:float = 0
    mean_accuracy:float = 0
    num_samples:int = 0
    global_step:int = 0

def do_epoch(
    model,
    loader,
    device,
    global_step:int,
    log_period:int = 500,
    max_steps:int = 0
) -> EpochMetrics:

    model.eval()
        
    epoch_loss = 0.0
    epoch_correct = 0
    epoch_samples = 0

    log_loss = 0.0
    log_correct = 0
    log_samples = 0

    step = 0

    for x, y in loader:

        if max_steps > 0 and step >= max_steps:
            break
        step += 1

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        with torch.no_grad():
            logits, encoding = model(x, bias=None)

            loss = F.cross_entropy(
                logits,
                y,
            )

            samples_step = x.size(0)
            loss_step  = (
                loss.item() * samples_step
            )
            epoch_loss += loss_step

            correct_step = (
                (logits.argmax(dim=1) == y)
                .sum()
                .item()
            )
            epoch_correct += correct_step
            epoch_samples += samples_step

            log_loss += loss_step
            log_correct += correct_step
            log_samples += samples_step

        if (log_samples % log_period) == 0:
            log_accuracy = log_correct / log_samples
            print(f"Step: {global_step} Accuracy: {log_accuracy}")
            log_loss = 0.0
            log_correct = 0
            log_samples = 0

        global_step += 1

    epoch_metrics = EpochMetrics(
        mean_loss = epoch_loss / epoch_samples,
        mean_accuracy =  epoch_correct / epoch_samples,
        num_samples = epoch_samples,
        global_step = global_step,
    )
    return epoch_metrics

def evaluate(
        exclude_classes_coarse:set[int]|None, 
        exclude_classes_fine:set, 
        model_file:str, 
        batch_size:int = 128, 
        training:bool = False
):
    dataset = Cifar100Dataset(
        file_path=data_file_path, 
        label_type=Cifar100Dataset.LABEL_TYPE_COARSE,
        training=training,
        exclude_classes_coarse=exclude_classes_coarse,
        exclude_classes_fine=exclude_classes_fine,
    )

    config = ResNetConfig(
        num_classes=dataset.get_num_classes(),
    )
    model = CifarClassifier(config, bias_stage=-1).to(device)

    state_dict = torch.load(model_file, weights_only=True)
    model.load_state_dict(state_dict)

    data_loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    )

    epoch_metrics = do_epoch(
        model = model,
        loader = data_loader,
        device = device,
        global_step = 0,
        max_steps = 0,
    )

    print(
        f"Epoch stats: "
        f"| loss {epoch_metrics.mean_loss:.4f} "
        f"| accuracy {epoch_metrics.mean_accuracy:.3f}"
    )



In [ ]:
classes_fine_12 = Cifar100Dataset.get_fine_classes([1,2])
classes_fine_345 = Cifar100Dataset.get_fine_classes([3,4,5])

classes_coarse_except_01 = {
    2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,
}

In [ ]:
model_file = '../cifar_100_pretrain/cifar_100_all_e9_64.25.pth'
#evaluate(exclude_classes_fine=classes_12, model_file=model_file)  # eval model=all on classes=345 accuracy 0.644


In [ ]:
#evaluate(exclude_classes_fine=classes_345, model_file=model_file)  # eval model=all on classes=12 accuracy 0.640


In [ ]:
model_file = '../cifar_100_pretrain/cifar_100_subclasses_12_e11_31.1.pth'


In [ ]:
evaluate(
    exclude_classes_coarse=classes_coarse_except_01, 
    exclude_classes_fine=classes_fine_12,  # i.e. test on 3, 4, 5
    model_file=model_file, 
    training=True,
)  

# Coarse 0,1
# Train Epoch stats: | loss 3.7232 | accuracy 0.316
# Eval Epoch stats: | loss 3.7013 | accuracy 0.330

# All coarse classes
# eval model=12 on classes=345 accuracy 0.311 train acc 31.5%


In [ ]:
evaluate(
    exclude_classes_coarse=classes_coarse_except_01, 
    exclude_classes_fine=classes_fine_345,  # i.e. test on 1,2
    model_file=model_file, 
    training=True,
)  

# Coarse 0,1
# Train Epoch stats: | loss 0.1135 | accuracy 0.967
# Eval Epoch stats: | loss 1.4608 | accuracy 0.625

# All coarse classes
# eval model=12 on classes=12 accuracy 0.636 train acc 94%
